# 📊 Superstore España: análisis de ventas ficticias

El notebook usa `data/superstore.csv`: **9.994 líneas de ventas sintéticas de España**, generadas con una semilla reproducible. Clientes ficticios, ciudades españolas, comunidades autónomas y productos en español; todos los importes están en **euros (EUR)**.

Los nombres técnicos de columnas se conservan para mantener compatibilidad con Reflex. Las regiones son agrupaciones comerciales de ejemplo. Las fechas internas mantienen el formato MM/DD/YYYY y se interpretan explícitamente al cargarlas.

La copia del dataset anterior está en `data/superstore_original.csv`. Para regenerar los datos usa `data/generar_superstore_es.py`.

Ejecuta el notebook desde el principio y reinicia el panel de filtros cuando cambies el CSV. Las conclusiones deben comprobarse para la selección actual; estos datos no describen el mercado real.

# 🛠️ Etapas del análisis

- **Limpieza:** revisar valores ausentes, duplicados y atípicos. El límite de ventas inferior a 5000 se puede activar en el panel; está desactivado por defecto. Los filtros pueden cambiar las conclusiones.
- **Análisis exploratorio (EDA):** estudiar distribuciones y comparar ventas y beneficio por categoría y región.
- **Aprendizaje automático:** entrenar modelos de regresión y evaluar MAE, RMSE y R².
- **Ingeniería de características:** crear precio por unidad, margen de beneficio y variables de calendario.
- **Visualización:** presentar gráficos por categoría, región, mes, estación y día de la semana.

Ejecuta las celdas en orden. Selecciona un kernel con NumPy, Pandas, Matplotlib, Seaborn y scikit-learn.

## Entorno local y ejecución

El archivo se busca en `data/superstore.csv`, situado junto a la carpeta `notebooks`. Ejecuta desde el principio después de seleccionar tu kernel de Python. Se han borrado las salidas antiguas para evitar mostrar gráficas en el idioma original.

Se mantienen los nombres de variables, columnas y categorías de los datos. Se traducen explicaciones, comentarios, títulos y ejes. El ejemplo conserva su planteamiento de modelado; revisa las notas sobre validación y posible fuga de información antes de interpretar las métricas como predicción de ventas futuras.

In [ ]:
# Entorno local: ya no dependemos de las rutas de Kaggle.
from pathlib import Path
import numpy as np
import pandas as pd

# Permite ejecutar desde notebooks/, desde el proyecto o desde EjerciciosPython.
DATA_PATH = None
for folder in [Path.cwd(), *Path.cwd().parents]:
    for candidate in [folder / 'data' / 'superstore.csv',
                      folder / 'Ejercicio991Reflex' / 'data' / 'superstore.csv']:
        if candidate.is_file():
            DATA_PATH = candidate
            break
    if DATA_PATH is not None:
        break
if DATA_PATH is None:
    raise FileNotFoundError('No se encuentra data/superstore.csv. Abre el notebook dentro del proyecto.')
print(f'Archivo de datos: {DATA_PATH}')

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings("ignore", category=FutureWarning)
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import GridSearchCV
from sklearn.ensemble import RandomForestRegressor

In [ ]:
df = pd.read_csv(DATA_PATH, encoding='ISO-8859-1', dtype={'Postal Code': str})
df.head()

In [ ]:
df.isnull().sum()

In [ ]:
df.info()

In [ ]:
df.columns.str.strip()

In [ ]:
df.columns = df.columns.str.replace(" ", "_").str.lower()
df.columns

In [ ]:
df['order_date'] = pd.to_datetime(df['order_date'], format='%m/%d/%Y')
df['ship_date'] = pd.to_datetime(df['ship_date'], format='%m/%d/%Y')

In [ ]:
df['postal_code'] = df['postal_code'].astype(str)

In [ ]:
# Contar filas duplicadas; la primera aparición no se marca.
df.duplicated().sum()

In [ ]:
df.describe()

In [ ]:
df.describe(include = "object").T

In [ ]:
df = df.drop_duplicates()

## 🎛️ Panel de filtros interactivos

Ejecuta esta celda **una vez** para conservar `df_base`, la copia completa anterior a los filtros. La celda siguiente crea controles con `ipywidgets` (requiere la extensión Jupyter de VS Code y un kernel activo).

Puedes combinar fechas de pedido y envío, año, mes, día, regiones, países, estados, ciudades, segmentos, modos de envío, categorías, subcategorías, clientes, productos, códigos e importes. **Todos los filtros se combinan con Y (AND)**; dentro de una selección múltiple se acepta cualquiera de los valores (OR). Una selección vacía significa «todos».

1. Ajusta los controles y pulsa **Aplicar filtros**.
2. Se actualizan `df`, los indicadores, la vista previa y cuatro gráficas del panel.
3. Para actualizar el resto del notebook, ejecuta **desde la primera celda posterior al panel hacia abajo**. No ejecutes de nuevo la celda de copia base ni la carga del CSV: restaurarían la selección.
4. **Restablecer** limpia los controles y recupera todos los registros base.

Los modelos originales necesitan suficientes filas: una selección vacía o muy pequeña sirve para explorar, pero no para entrenar modelos. Las conclusiones escritas del cuaderno original no cambian automáticamente con los filtros.

In [ ]:
# Instantánea independiente: los filtros siempre parten de aquí, no del último df filtrado.
df_base = df.copy(deep=True)
print(f'Base disponible: {len(df_base):,} filas. No se modifica el CSV.')

In [ ]:
import ipywidgets as widgets
from IPython.display import display, clear_output

# Función pura: permite probar los filtros sin depender de la interfaz.
# Función compartida: si cambias sus reglas, notebook y Reflex cambian juntos.
import sys
project_path = DATA_PATH.parent.parent
if str(project_path) not in sys.path:
    sys.path.insert(0, str(project_path))
from Ejercicio991Reflex.superstore_analysis import filtrar_superstore

style = {'description_width': 'initial'}
layout = widgets.Layout(width='360px')
category_names = {'region':'Regiones', 'country':'Países', 'state':'Comunidades autónomas',
                  'city':'Ciudades', 'segment':'Segmentos', 'ship_mode':'Modos de envío',
                  'category':'Categorías', 'sub-category':'Subcategorías'}
category_controls = {
    col: widgets.SelectMultiple(options=sorted(df_base[col].dropna().unique()),
                                description=label, rows=5, style=style, layout=layout)
    for col, label in category_names.items()}
text_names = {'customer_name':'Nombre de cliente', 'product_name':'Nombre de producto',
              'customer_id':'ID de cliente', 'product_id':'ID de producto',
              'order_id':'ID de pedido', 'postal_code':'Código postal'}
text_controls = {col: widgets.Text(description=label, placeholder='Contiene…', style=style, layout=layout)
                 for col, label in text_names.items()}
range_names = {'sales':'Ventas', 'profit':'Beneficio', 'quantity':'Cantidad', 'discount':'Descuento (0–1)'}
range_controls = {}
for col, label in range_names.items():
    lo, hi = float(df_base[col].min()), float(df_base[col].max())
    range_controls[col] = widgets.FloatRangeSlider(
        value=(lo, hi), min=lo, max=hi, step=1 if col=='quantity' else .001 if col=='discount' else .01,
        description=label, continuous_update=False, readout_format='.2f', style=style,
        layout=widgets.Layout(width='650px', max_width='100%'))
years = widgets.SelectMultiple(options=sorted(df_base['order_date'].dt.year.unique().tolist()), description='Años', style=style)
months = widgets.SelectMultiple(options=list(enumerate([])), description='Meses', style=style)
months.options = [(name, i+1) for i,name in enumerate(['Enero','Febrero','Marzo','Abril','Mayo','Junio','Julio','Agosto','Septiembre','Octubre','Noviembre','Diciembre'])]
days = widgets.SelectMultiple(options=[(x,i) for i,x in enumerate(['Lunes','Martes','Miércoles','Jueves','Viernes','Sábado','Domingo'])], description='Días', style=style)
date_controls = {col:(widgets.DatePicker(description=label+' desde', style=style), widgets.DatePicker(description=label+' hasta', style=style))
                 for col,label in [('order_date','Pedido'),('ship_date','Envío')]}
profit_control = widgets.Dropdown(options=['Todas','Con beneficio','Con pérdidas','Sin beneficio ni pérdida'], description='Rentabilidad', style=style)
no_discount = widgets.Checkbox(description='Solo ventas sin descuento')
limit = widgets.Checkbox(description='Solo ventas inferiores a 5000')
duplicates = widgets.Checkbox(description='Eliminar duplicados', value=True)
apply_button = widgets.Button(description='Aplicar filtros', button_style='primary', icon='filter')
reset_button = widgets.Button(description='Restablecer', icon='refresh')
output = widgets.Output()

def recoger_filtros():
    return {'categorias':{col:list(control.value) for col,control in category_controls.items()},
            'textos':{col:control.value for col,control in text_controls.items()},
            'rangos':{col:control.value for col,control in range_controls.items()},
            'fechas':{col:(a.value,b.value) for col,(a,b) in date_controls.items()},
            'años':list(years.value), 'meses':list(months.value), 'dias':list(days.value),
            'rentabilidad':profit_control.value, 'sin_descuento':no_discount.value,
            'limite_5000':limit.value, 'quitar_duplicados':duplicates.value}

def aplicar_filtros(_=None):
    global df  # El resto del notebook trabaja con este DataFrame filtrado.
    config = recoger_filtros()
    with output:
        clear_output(wait=True)
        for start,end in config['fechas'].values():
            if start and end and start > end:
                print('La fecha inicial no puede ser posterior a la final. Se conserva la selección anterior.')
                return
        df = filtrar_superstore(df_base, config)
        print(f'Selección: {len(df):,} / {len(df_base):,} filas. Ejecuta las celdas posteriores para actualizar sus análisis.')
        if df.empty:
            print('No hay coincidencias. Amplía los filtros o pulsa Restablecer.')
            return
        display(pd.DataFrame({'Filas':[len(df)], 'Ventas':[round(df.sales.sum(),2)],
                              'Beneficio':[round(df.profit.sum(),2)], 'Unidades':[df.quantity.sum()]}))
        display(df.head(20))
        fig, axes = plt.subplots(2,2,figsize=(14,9),constrained_layout=True)
        df.groupby('region')['sales'].sum().plot.bar(ax=axes[0,0], title='Ventas por región', color='#6366f1')
        df.groupby('category')['profit'].sum().plot.bar(ax=axes[0,1], title='Beneficio por categoría', color='#10b981')
        df.resample('ME', on='order_date')['sales'].sum().plot(ax=axes[1,0], title='Ventas mensuales', color='#6366f1')
        axes[1,1].scatter(df['discount'],df['profit'], alpha=.25, color='#f59e0b')
        axes[1,1].set(title='Descuento y beneficio', xlabel='Descuento', ylabel='Beneficio')
        for ax in axes.flat: ax.tick_params(axis='x', rotation=30)
        plt.show()

def restablecer_filtros(_=None):
    for control in category_controls.values(): control.value=()
    for control in text_controls.values(): control.value=''
    for control in range_controls.values(): control.value=(control.min,control.max)
    for pair in date_controls.values():
        for control in pair: control.value=None
    years.value=months.value=days.value=()
    profit_control.value='Todas'
    no_discount.value=limit.value=False
    duplicates.value=True
    aplicar_filtros()

apply_button.on_click(aplicar_filtros)
reset_button.on_click(restablecer_filtros)
panels = [
 widgets.VBox([widgets.HTML('<b>Selección múltiple: Ctrl/Cmd + clic. Sin selección = todos.</b>'),
               widgets.Box(list(category_controls.values()),layout=widgets.Layout(display='flex',flex_flow='row wrap',gap='12px'))]),
 widgets.VBox([widgets.HBox([years,months,days]), *[widgets.HBox(list(pair)) for pair in date_controls.values()]]),
 widgets.VBox(list(range_controls.values())+[profit_control,no_discount,limit,duplicates]),
 widgets.Box(list(text_controls.values()),layout=widgets.Layout(display='flex',flex_flow='row wrap',gap='12px'))]
accordion = widgets.Accordion(children=panels)
for i,title in enumerate(['Ubicación, categorías y segmentos','Fechas y calendario','Importes y rentabilidad','Búsqueda de clientes, productos e identificadores']): accordion.set_title(i,title)
display(widgets.VBox([accordion,widgets.HBox([apply_button,reset_button]),output]))
aplicar_filtros()

### Continuar con los análisis de la selección

Tras aplicar filtros, ejecuta desde la siguiente celda hacia abajo. Esa celda elimina identificadores del `df` usado para los modelos; `df_base` conserva los datos completos para poder filtrar de nuevo. Las conclusiones del texto son orientativas y no se recalculan.

**Ejemplos:** región West + categoría Technology; año 2017 + solo pérdidas; ventas entre 100 y 1000 + descuento inferior a 0,20. Los límites de fechas e importes son inclusivos; el recorte opcional de 5000 es estricto.

In [ ]:
# Excluir identificadores y nombres del conjunto usado para el modelo.
drop_cols = ['row_id', 'order_id', 'customer_name', 'product_name', 'postal_code', 'customer_id', 'product_id']
df = df.drop(columns=drop_cols)

In [ ]:
# Los límites de ventas ahora se eligen en el panel de filtros.
# No volvemos a recortar df aquí: todas las gráficas deben usar la misma selección.
df['sales'].describe()
if not df.empty:
    df['sales'].plot(kind='box', title='Ventas de la selección actual')
    plt.show()

In [ ]:
corr = df.corr(numeric_only=True)
plt.figure(figsize=(10,6))
sns.heatmap(corr, annot=True, cmap='coolwarm')
plt.title("Matriz de correlación")
plt.show()

# Análisis exploratorio de datos (EDA)

# 📊 Distribución de ventas y beneficio

Analiza la asimetría de `sales` y los valores positivos y negativos de `profit` en el CSV español sintético. Los importes se expresan en euros. Los filtros cambian la población analizada.

In [ ]:
# Configurar el tema visual.
sns.set_theme(style="whitegrid", palette="pastel")

# Crear dos gráficos.
fig, axs = plt.subplots(1, 2, figsize=(14, 5))

# Distribución de ventas con log1p y estimación de densidad.
sns.histplot(np.log1p(df['sales']), kde=True, ax=axs[0], color='skyblue')
axs[0].set_title('Distribución de ventas (transformación logarítmica)', fontsize=13)
axs[0].set_xlabel('Log(1 + importe de ventas)')
axs[0].set_ylabel('Frecuencia')

# Distribución del beneficio en su escala original.
sns.histplot(df['profit'], kde=True, ax=axs[1], color='lightgreen')
axs[1].set_title('Distribución del beneficio', fontsize=13)
axs[1].set_xlabel('Importe del beneficio')
axs[1].set_ylabel('Frecuencia')

plt.tight_layout()
plt.show()


### 💰 Observaciones sobre las ventas

- La distribución de los importes tiene una cola hacia valores altos.
- Hay muchas operaciones pequeñas y algunas grandes que elevan la facturación total.
- Para conocer su contribución real conviene comparar tanto el número de operaciones como sus importes.

💡 **Propuesta:** estudiar campañas para productos de mayor precio y evaluar conjuntamente volumen y margen en los de menor precio.

### 📉 Distribución del beneficio

Hay operaciones positivas y negativas. Los importes de pérdidas del CSV anterior ya no son aplicables. Comprueba el mínimo, los percentiles y la distribución de la selección actual. El beneficio bruto sintético depende del precio, las unidades, el descuento y el coste de compra.

In [ ]:
plt.figure(figsize=(10,5))
sns.barplot(data=df, x='ship_mode', y='sales', estimator='mean', palette='Blues_d')
plt.title('Ventas medias por modo de envío')
plt.xlabel('Modo de envío')
plt.ylabel('Ventas medias')
plt.show()

In [ ]:
plt.figure(figsize=(10,5))
sns.barplot(data=df, x='ship_mode', y='profit', estimator='mean', palette='Greens_d')
plt.title('Beneficio medio por modo de envío')
plt.xlabel('Modo de envío')
plt.ylabel('Beneficio medio')
plt.show()


### 🚚 Ventas y beneficio según el modo de envío

Se comparan los valores medios de ventas y beneficio por modalidad de envío.

El envío en el mismo día (`Same Day`) podría asociarse con pedidos urgentes de mayor importe. Su posible coste adicional debe comprobarse con datos de costes; no puede deducirse únicamente de estas medias.

💡 **Propuesta:** revisar los descuentos y la rentabilidad de los pedidos con entrega rápida.

In [ ]:
plt.figure(figsize=(8,5))
sns.barplot(data=df, x='segment', y='sales', estimator='mean', palette='Oranges')
plt.title('Ventas medias por segmento')
plt.xlabel('Segmento de cliente')
plt.ylabel('Ventas medias')
plt.show()


In [ ]:
plt.figure(figsize=(8,5))
sns.barplot(data=df, x='segment', y='profit', estimator='mean', palette='Purples')
plt.title('Beneficio medio por segmento')
plt.xlabel('Segmento de cliente')
plt.ylabel('Beneficio medio')
plt.show()

### 👥 Ventas y beneficio por segmento

Compara los segmentos **Particular**, **Empresa** y **Oficina en casa** del conjunto sintético español. Ordena las medias actuales antes de afirmar qué grupo vende más o tiene mayor beneficio. Los resultados cambian al aplicar filtros y no representan el mercado español real.

In [ ]:
plt.figure(figsize=(8,5))
sns.barplot(data=df, x='category', y='profit', estimator='mean', palette='Set2')
plt.title('Beneficio medio por categoría')
plt.xlabel('Categoría')
plt.ylabel('Beneficio medio')
plt.show()

In [ ]:
plt.figure(figsize=(14,6))
sns.barplot(data=df, x='sub-category', y='profit', estimator='mean', palette='Set3')
plt.title('Beneficio medio por subcategoría')
plt.xlabel('Subcategoría')
plt.ylabel('Beneficio medio')
plt.xticks(rotation=45)
plt.show()


### 📦 Beneficio por categoría y subcategoría

El CSV español contiene **Mobiliario**, **Material de oficina** y **Tecnología**. Examina qué subcategorías tienen beneficios positivos o negativos. Las pérdidas surgen cuando los descuentos hacen que la venta quede por debajo del coste sintético de compra. Las conclusiones deben basarse en la selección actual.

In [ ]:
region_sales = df.groupby('region')['sales'].sum()

region_sales.plot(kind='pie', autopct='%1.1f%%', figsize=(6,6), title='Distribución de ventas por región')
plt.ylabel('')
plt.show()


In [ ]:
plt.figure(figsize=(10,5))
sns.boxplot(data=df, x='region', y='profit')
plt.title('Distribución del beneficio por región', fontsize=13)
plt.xlabel('Región')
plt.ylabel('Beneficio')
plt.grid(True)
plt.show()


### 🌍 Ventas y beneficio por región comercial

Las ciudades y comunidades autónomas se agrupan en regiones comerciales de ejemplo: Centro, Este, Norte, Noreste, Noroeste, Sur, Sureste, Oeste e Islas. No son una clasificación estadística oficial.

Compara las proporciones de ventas y la distribución del beneficio. Los porcentajes anteriores del dataset estadounidense ya no se aplican; utiliza los resultados actuales de las gráficas.

In [ ]:
# Calcular las ventas totales mensuales.
monthly_sales = df.resample('ME', on='order_date')['sales'].sum()

# Dibujar la serie temporal.
plt.figure(figsize=(14,6))
monthly_sales.plot(color='royalblue', linewidth=2)
plt.title('Evolución mensual de las ventas totales')
plt.xlabel('Fecha')
plt.ylabel('Ventas totales')
plt.grid(True)
plt.show()


### 📅 Evolución mensual de las ventas

La línea muestra cómo cambia la facturación total cada mes.

- Los picos pueden coincidir con campañas o fechas especiales, pero esa relación requiere comprobación.
- Los descensos pueden reflejar estacionalidad o cambios de demanda.
- Una tendencia ascendente indica crecimiento de la facturación en el periodo analizado.

💡 **Propuestas:** planificar existencias para los meses de mayor demanda y estudiar campañas en los de menor actividad. Una predicción temporal requeriría además validación cronológica.

In [ ]:
y = df['sales']
X = df[['category', 'sub-category', 'segment', 'discount', 'quantity', 'profit', 'region', 'ship_mode']]

# Separar variables categóricas y numéricas.
categorical_cols = ['category', 'sub-category', 'segment', 'region', 'ship_mode']
numeric_cols = ['discount', 'quantity', 'profit']

# Codificar categorías con OneHotEncoder.
preprocessor = ColumnTransformer(
    transformers=[
        ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_cols)
    ],
    remainder='passthrough'  # Conservar las columnas numéricas sin transformarlas.
)

# Dividir en entrenamiento y validación.
X_train, X_valid, y_train, y_valid = train_test_split(X, y, test_size=0.2, random_state=42)

In [ ]:
# Crear el flujo de preprocesamiento y regresión.
modelL = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('regressor', LinearRegression())
])

# Entrenar el modelo.
modelL.fit(X_train, y_train)

In [ ]:
# Generar predicciones.
y_pred = modelL.predict(X_valid)

# Calcular métricas de evaluación.
mae = mean_absolute_error(y_valid, y_pred)
rmse = np.sqrt(mean_squared_error(y_valid, y_pred))
r2 = r2_score(y_valid, y_pred)

print(f"MAE: {mae:.2f}")
print(f"RMSE: {rmse:.2f}")
print(f"R²: {r2:.2f}")

In [ ]:
y = np.log1p(df['sales'])  # Calcular log1p del objetivo; esta asignación no cambia y_train creado anteriormente.
model = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('regressor', RandomForestRegressor(random_state=42))
])
model.fit(X_train, y_train)

In [ ]:
y_pred = model.predict(X_valid)

# Si entrenas con un objetivo logarítmico, aplica la transformación inversa.
# y_valid = np.expm1(y_valid)
# y_pred = np.expm1(y_pred)

# Calcular las métricas de evaluación.
mae = mean_absolute_error(y_valid, y_pred)
rmse = np.sqrt(mean_squared_error(y_valid, y_pred))
r2 = r2_score(y_valid, y_pred)

print(f"MAE: {mae:.2f}")
print(f"RMSE: {rmse:.2f}")
print(f"R²: {r2:.4f}")

In [ ]:
# Obtener los nombres de las características después del preprocesamiento.
feature_names = model.named_steps['preprocessor'].get_feature_names_out()

In [ ]:
# Obtener la importancia de las características del bosque aleatorio.
importances = model.named_steps['regressor'].feature_importances_


In [ ]:
# Crear un DataFrame de importancias.
fi_df = pd.DataFrame({'Feature': feature_names, 'Importance': importances})

# Seleccionar las diez características principales.
fi_df = fi_df.sort_values('Importance', ascending=False).head(10)

# Dibujar el gráfico.
plt.figure(figsize=(10,6))
plt.barh(fi_df['Feature'], fi_df['Importance'], color='skyblue')
plt.gca().invert_yaxis()
plt.title('Importancia de las características (bosque aleatorio)')
plt.xlabel('Importancia')
plt.ylabel('Características')
plt.show()

### 🎯 Importancia de las características

El gráfico muestra qué variables utiliza más el bosque aleatorio: beneficio (`profit`), descuento (`discount`), cantidad (`quantity`) y categorías de producto.

La importancia del modelo no demuestra causalidad. Además, `profit` puede no estar disponible antes de realizar la venta y está relacionado con su importe: revisa el riesgo de fuga de información antes de utilizar este modelo para predicción real.

## Entrenar un modelo reducido con las características principales

In [ ]:
# Reducir X a las cinco características seleccionadas.
X_top = df[['profit', 'discount', 'quantity', 'sub-category', 'category']]

# Conservar el objetivo de ventas.
y = df['sales']

In [ ]:
# Definir las nuevas listas de variables.
categorical_cols_top = ['sub-category', 'category']
numeric_cols_top = ['profit', 'discount', 'quantity']

# Crear el nuevo preprocesador.
preprocessor_top = ColumnTransformer(
    transformers=[
        ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_cols_top)
    ],
    remainder='passthrough'
)

In [ ]:
# Dividir los datos.
X_train_top, X_valid_top, y_train_top, y_valid_top = train_test_split(X_top, y, test_size=0.2, random_state=42)

# Crear el modelo reducido.
model_top = Pipeline(steps=[
    ('preprocessor', preprocessor_top),
    ('regressor', RandomForestRegressor(random_state=42))
])

# Entrenarlo.
model_top.fit(X_train_top, y_train_top)

In [ ]:
# Generar predicciones.
y_pred_top = model_top.predict(X_valid_top)

# Calcular las métricas.
mae_top = mean_absolute_error(y_valid_top, y_pred_top)
rmse_top = np.sqrt(mean_squared_error(y_valid_top, y_pred_top))
r2_top = r2_score(y_valid_top, y_pred_top)

print(f"MAE del modelo reducido: {mae_top:.2f}")
print(f"RMSE del modelo reducido: {rmse_top:.2f}")
print(f"R² del modelo reducido: {r2_top:.4f}")

El modelo reducido utiliza cinco características. Compara sus valores de MAE, RMSE y R² con los del modelo completo para comprobar si mejora: utilizar menos variables no garantiza por sí solo un mejor resultado.

In [ ]:
# Recalcular las predicciones del modelo completo.
y_pred = model.predict(X_valid)

# Diagrama de dispersión.
plt.figure(figsize=(8,6))
plt.scatter(y_valid, y_pred, alpha=0.5, color='skyblue')
plt.plot([y_valid.min(), y_valid.max()], [y_valid.min(), y_valid.max()], '--r')
plt.xlabel('Ventas reales')
plt.ylabel('Ventas predichas')
plt.title('Ventas predichas frente a reales (modelo completo)')
plt.show()

### 📈 Predicciones frente a ventas reales del conjunto sintético

La diagonal roja representa una predicción perfecta. Comprueba si los errores aumentan con el importe de la venta y si hay suficientes operaciones grandes en la selección. Las conclusiones del modelo anterior deben recalcularse con estos nuevos datos.

# Ingeniería de características

In [ ]:
# Ventas / cantidad: precio por unidad.
df['price_per_unit'] = df['sales'] / df['quantity']


In [ ]:
# Beneficio / ventas: margen de beneficio.
df['profit_margin'] = df['profit'] / df['sales']
df['profit_margin'] = df['profit_margin'].replace([np.inf, -np.inf], 0).fillna(0)


In [ ]:
# Indicador de descuento superior al 50 %.
df['high_discount_flag'] = (df['discount'] > 0.5).astype(int)


In [ ]:
# Día del pedido: 0 = lunes, 6 = domingo.
df['order_date'] = pd.to_datetime(df['order_date'])  # Convertir a fecha si aún no lo es.
df['order_day'] = df['order_date'].dt.dayofweek


In [ ]:
# Extraer el mes del pedido.
df['order_month'] = df['order_date'].dt.month


In [ ]:
# Asignar una estación al mes del pedido (hemisferio norte).
def get_season(month):
    if month in [12, 1, 2]:
        return 'Invierno'
    elif month in [3, 4, 5]:
        return 'Primavera'
    elif month in [6, 7, 8]:
        return 'Verano'
    else:
        return 'Otoño'

df['order_season'] = df['order_month'].apply(get_season)


In [ ]:
df.head()

# Visualización de datos

In [ ]:
# 1. Agrupar ventas y beneficio por categoría.
category_summary = df.groupby('category').agg({'sales': 'sum', 'profit': 'sum'}).reset_index()

# 2. Gráfico de ventas totales.
plt.figure(figsize=(12,6))
sns.barplot(data=category_summary, x='category', y='sales', palette='Blues_d')
plt.title('Ventas totales por categoría', fontsize=16)
plt.xlabel('Categoría', fontsize=14)
plt.ylabel('Ventas totales', fontsize=14)
plt.xticks(fontsize=12)
plt.yticks(fontsize=12)
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.show()

# 3. Gráfico de beneficio total.
plt.figure(figsize=(12,6))
sns.barplot(data=category_summary, x='category', y='profit', palette='Greens_d')
plt.title('Beneficio total por categoría', fontsize=16)
plt.xlabel('Categoría', fontsize=14)
plt.ylabel('Beneficio total', fontsize=14)
plt.xticks(fontsize=12)
plt.yticks(fontsize=12)
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.show()

In [ ]:
# Sumar ventas por región.
region_sales = df.groupby('region')['sales'].sum()

# Dibujar el gráfico circular.
plt.figure(figsize=(8,8))
colors = sns.color_palette('pastel')[0:4] 

plt.pie(region_sales, labels=region_sales.index, colors=colors, autopct='%1.1f%%', startangle=140)
plt.title('Distribución de ventas por región', fontsize=16, fontweight='bold')
plt.axis('equal')  # Mantener el círculo sin deformaciones.
plt.show()


In [ ]:
# Sumar ventas por mes.
monthly_sales = df.groupby(df['order_date'].dt.to_period('M')).agg({'sales': 'sum'}).reset_index()
monthly_sales['order_date'] = monthly_sales['order_date'].astype(str)  # Convertir los periodos en texto.

# Dibujar la serie temporal.
plt.figure(figsize=(14,6))
sns.lineplot(data=monthly_sales, x='order_date', y='sales', marker='o', color='royalblue')
plt.title('Evolución mensual de las ventas', fontsize=16, fontweight='bold')
plt.xlabel('Mes', fontsize=14)
plt.ylabel('Ventas totales (€)', fontsize=14)
plt.xticks(rotation=45, fontsize=10)
plt.yticks(fontsize=12)
plt.grid(True, linestyle='--', alpha=0.7)
plt.tight_layout()
plt.show()


In [ ]:
plt.figure(figsize=(16,8))
sns.boxplot(data=df, x='sub-category', y='profit', palette='coolwarm')
plt.title('Distribución del beneficio por subcategoría', fontsize=16, fontweight='bold')
plt.xlabel('Subcategoría', fontsize=14)
plt.ylabel('Beneficio (€)', fontsize=14)
plt.xticks(rotation=45, fontsize=12)
plt.yticks(fontsize=12)
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(10,6))
sns.scatterplot(data=df, x='discount', y='profit', alpha=0.6, color='darkorange')
plt.title('Relación entre descuento y beneficio', fontsize=16, fontweight='bold')
plt.xlabel('Proporción de descuento', fontsize=14)
plt.ylabel('Beneficio (€)', fontsize=14)
plt.grid(True, linestyle='--', alpha=0.7)
plt.show()


In [ ]:
plt.figure(figsize=(10,6))
sns.boxplot(data=df, x='order_season', y='sales', palette='Set3')
plt.title('Distribución de ventas por estación', fontsize=16, fontweight='bold')
plt.xlabel('Estación', fontsize=14)
plt.ylabel('Ventas (€)', fontsize=14)
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.show()

In [ ]:
df['order_day'] = df['order_date'].dt.dayofweek  # Lunes = 0, domingo = 6.

day_map = {0: 'Lunes', 1: 'Martes', 2: 'Miércoles', 3: 'Jueves', 4: 'Viernes', 5: 'Sábado', 6: 'Domingo'}
df['order_day_name'] = df['order_day'].map(day_map)
day_sales = df.groupby('order_day_name')['sales'].mean().reindex(['Lunes', 'Martes', 'Miércoles', 'Jueves', 'Viernes', 'Sábado', 'Domingo'])

# Dibujar el gráfico.
plt.figure(figsize=(12,6))
sns.lineplot(x=day_sales.index, y=day_sales.values, marker='o', color='mediumseagreen')
plt.title('Ventas medias por día de la semana', fontsize=16, fontweight='bold')
plt.xlabel('Día', fontsize=14)
plt.ylabel('Ventas medias (€)', fontsize=14)
plt.xticks(fontsize=12)
plt.yticks(fontsize=12)
plt.grid(True, linestyle='--', alpha=0.7)
plt.show()


# 📋 Conclusiones de la selección actual

El CSV contiene ventas ficticias de España y valores en EUR. No son cifras oficiales ni transacciones reales. Los clientes, productos, cantidades, descuentos y fechas se generan con una semilla reproducible.

Revisa qué categorías, regiones, periodos y segmentos destacan tras aplicar tus filtros. Las conclusiones escritas para el dataset estadounidense se han retirado porque ya no describen estos datos.

Beneficio bruto = ventas después de descuento − coste sintético de compra. No incluye IVA, transporte, alquileres, salarios ni otros gastos. Los modelos son ejercicios exploratorios y no predicciones del mercado español.